# 13 · Predicciones por lotes (capa batch)

**Propósito:** aplicar los modelos finales a los vuelos y publicar el **nivel de alerta** de cada salida:

| Nivel | Regla |
|---|---|
| rojo | probabilidad de 60 min o más ≥ umbral rojo |
| ámbar | no es rojo y probabilidad de 15 min o más ≥ umbral ámbar |
| verde | ninguna de las anteriores |

Es la capa **por lotes** de la arquitectura: se corre una vez por mes cuando llega el VRA nuevo (o para un mes
puntual con `MES`). No reentrena: carga el `PipelineModel` guardado y llama a `transform`.

**Entrada:** `gold/features_salida`, `gold/modelos/final_*` y `gold/modelos/umbrales_final`.

**Salida:** `gold/predicciones/pais=XX/year=AAAA/month=M/` (solo se reemplazan las particiones calculadas).

**Requisito:** notebook 12.

In [ ]:
import os
from pathlib import Path

import yaml

# Carpeta del repositorio (este notebook está en notebooks/)
RAIZ = Path.cwd().parent
with open(RAIZ / "config" / "pipeline.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
tablas = cfg["lake"]["tablas"]

# En la EC2 la variable LAKE_BUCKET trae el nombre del bucket; en una laptop se trabaja sobre data/lake
BUCKET = os.environ.get("LAKE_BUCKET", "").strip()
RAIZ_LOCAL = RAIZ / cfg["lake"]["raiz_local"]
BASE = f"s3a://{BUCKET}" if BUCKET else RAIZ_LOCAL.resolve().as_uri()


def ruta(tabla: str) -> str:
    """Ruta completa de una tabla declarada en config/pipeline.yaml."""
    return f"{BASE}/{tablas[tabla]}"


def meses(desde: str, hasta: str) -> list[tuple[int, int]]:
    """Pares (año, mes) entre dos fechas AAAA-MM, ambas incluidas."""
    anio, mes = map(int, desde.split("-"))
    anio_fin, mes_fin = map(int, hasta.split("-"))
    salida = []
    while (anio, mes) <= (anio_fin, mes_fin):
        salida.append((anio, mes))
        anio, mes = (anio + 1, 1) if mes == 12 else (anio, mes + 1)
    return salida


DESDE, HASTA = cfg["periodo"]["inicio"], cfg["periodo"]["fin"]
print("Destino:", f"s3://{BUCKET}" if BUCKET else "data/lake (local)")
print("Periodo:", DESDE, "a", HASTA)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array

# En la EC2 Spark toma la configuración del clúster (spark-defaults.conf); en una laptop corre en modo local
constructor = SparkSession.builder.appName("latam-bigdata-predicciones")
if not BUCKET:
    constructor = (
        constructor.master("local[*]")
        .config("spark.sql.shuffle.partitions", str(cfg["spark"]["particiones_shuffle_local"]))
        .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "3g"))
    )
spark = (
    constructor.config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")


def en_periodo(df, col_anio="year", col_mes="month"):
    """Deja solo las filas cuyo año y mes están dentro del periodo de config/pipeline.yaml."""
    ym = F.format_string("%04d-%02d", F.col(col_anio), F.col(col_mes))
    return df.where((ym >= DESDE) & (ym <= HASTA))

In [ ]:
# Parámetros: qué particiones puntuar, o un mes puntual ("AAAA-MM") para la actualización mensual
PARTICIONES = ["valid", "test"]
MES = None

NUMERICAS = list(cfg["features"]["numericas"])
CATEGORICAS = list(cfg["features"]["categoricas"])
MODELOS_DIR = f"{BASE}/{tablas['gold_modelos']}"
EVID = RAIZ / "docs" / "evidencias" / "modelo"
EVID.mkdir(parents=True, exist_ok=True)

modelos = {o: PipelineModel.load(f"{MODELOS_DIR}/final_{o}") for o in ("y_demora15", "y_demora60")}
umbrales = {r.objetivo: r for r in spark.read.json(f"{MODELOS_DIR}/umbrales_final").collect()}
U_AMBAR, U_ROJO = umbrales["y_demora15"].umbral, umbrales["y_demora60"].umbral
VERSION = umbrales["y_demora15"].version
print(f"Modelo {VERSION} · umbral ámbar {U_AMBAR} · umbral rojo {U_ROJO}")

In [ ]:
def preparar(df):
    """Las mismas columnas que en el notebook 12: numéricas como double y categóricas como texto."""
    return df.select(
        "id_vuelo", "particion", "pais", "es_latam", "origen", "fecha_local", "year", "month",
        *[F.col(c).cast("double").alias(c) for c in NUMERICAS],
        *[F.coalesce(F.col(c).cast("string"), F.lit("desconocido")).alias(f"cat_{c}") for c in CATEGORICAS],
    )


def puntuar(df, modelo, nombre):
    """Aplica un PipelineModel y deja solo la probabilidad de demora con el nombre pedido."""
    salida = modelo.transform(df).withColumn(nombre, vector_to_array("probability")[1])
    return salida.drop(*[c for c in salida.columns if c not in df.columns and c != nombre])


gold = spark.read.parquet(ruta("gold_features"))
if MES:
    anio, mes = map(int, MES.split("-"))
    gold = gold.where((F.col("year") == anio) & (F.col("month") == mes))
else:
    gold = gold.where(F.col("particion").isin(PARTICIONES))

pred = puntuar(puntuar(preparar(gold), modelos["y_demora15"], "prob_demora15"), modelos["y_demora60"], "prob_demora60")
pred = pred.select(
    "id_vuelo", "particion", "es_latam", "origen", "fecha_local",
    F.round("prob_demora15", 5).alias("prob_demora15"), F.round("prob_demora60", 5).alias("prob_demora60"),
    F.lit(U_AMBAR).alias("umbral_ambar"), F.lit(U_ROJO).alias("umbral_rojo"),
    F.when(F.col("prob_demora60") >= U_ROJO, "rojo").when(F.col("prob_demora15") >= U_AMBAR, "ambar")
    .otherwise("verde").alias("nivel_alerta"),
    F.lit(VERSION).alias("version_modelo"), "pais", "year", "month",
)
(
    pred.repartition("pais", "year", "month")
    .write.mode("overwrite").partitionBy("pais", "year", "month").parquet(ruta("gold_predicciones"))
)
print("gold/predicciones escrito")

## Verificación: cuántas alertas por día recibiría el CCO

In [ ]:
pred = spark.read.parquet(ruta("gold_predicciones"))
resumen = (
    pred.where(F.col("es_latam") & (F.col("pais") == "BR"))
    .groupBy("particion", "nivel_alerta").agg(F.count("*").alias("vuelos"))
    .join(pred.where(F.col("es_latam") & (F.col("pais") == "BR")).groupBy("particion")
          .agg(F.countDistinct("fecha_local").alias("dias")), "particion")
    .withColumn("por_dia", F.round(F.col("vuelos") / F.col("dias"), 1))
    .orderBy("particion", "nivel_alerta").toPandas()
)
resumen.to_csv(EVID / "predicciones_resumen.csv", index=False)
resumen

In [ ]:
spark.stop()